In [ ]:
# Modified from notebooks_colab_py/119_baidu__Qianfan-OCR.py: the remote test image
# URL is replaced with a locally-downloaded copy (via hf-mirror), pre-decoded with PIL
# and passed as an image object so it runs fully offline and avoids torchvision's
# libjpeg-dependent JPEG decoder.
import subprocess as _nbsp

# ===== code cell =====
_nbsp.run(r'''pip install -U transformers''', shell=True)

# ===== code cell =====
# Use a pipeline as a high-level helper
from transformers import pipeline
from PIL import Image

from io import BytesIO
import requests
_img = Image.open(BytesIO(requests.get("https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers_logo_small.png").content)).convert("RGB")
pipe = pipeline("image-text-to-text", model="baidu/Qianfan-OCR")
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": _img},
            {"type": "text", "text": "What animal is on the candy?"}
        ]
    },
]
print(pipe(text=messages))

# ===== code cell =====
# Load model directly
from transformers import AutoProcessor, AutoModelForMultimodalLM

processor = AutoProcessor.from_pretrained("baidu/Qianfan-OCR")
model = AutoModelForMultimodalLM.from_pretrained("baidu/Qianfan-OCR")
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": _img},
            {"type": "text", "text": "What animal is on the candy?"}
        ]
    },
]
inputs = processor.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(processor.decode(outputs[0][inputs["input_ids"].shape[-1]:]))
